# 03 - Feature Engineering

Create memory-efficient time-series features for Temporal Fusion Transformer
(TFT) and PatchTST multi-horizon air-quality forecasting.

Forecasting design:

- Historical context: 168 hours
- Forecast horizon: 15 hours
- Models: TFT and PatchTST
- Forecast targets: 9 pollutant concentrations

This notebook creates:

- Stable station identifiers
- Global hourly time index
- Calendar features
- Cyclical time features
- Station-wise time-gap diagnostics
- Sequence continuity metadata
- Memory-optimized model-ready dataset

Manual lag features, rolling features, target shifts, scaling, and sequence
windows are intentionally not created here.

# 1. Import, Configureation and Paths

In [1]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "Data"
INTERIM_DATA_DIR = DATA_DIR / "Interim"
PROCESSED_DATA_DIR = DATA_DIR / "Processed"

INPUT_FILE = (
    INTERIM_DATA_DIR
    / "cleaned_air_quality.parquet"
)

OUTPUT_FILE = (
    PROCESSED_DATA_DIR
    / "engineered_air_quality.parquet"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ENCODER_LENGTH = 168
PREDICTION_LENGTH = 15

POLLUTANT_COLUMNS = [
    "pm2_5",
    "pm10",
    "no",
    "no2",
    "nox",
    "nh3",
    "co",
    "so2",
    "o3",
]

WEATHER_COLUMNS = [
    "ambient_temperature",
    "relative_humidity",
    "solar_radiation",
    "rainfall",
]

STATION_KEYS = [
    "state",
    "city",
    "latitude",
    "longitude",
]

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

print(f"\nInput file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

print(f"\nEncoder length: {ENCODER_LENGTH} hours")
print(f"Prediction length: {PREDICTION_LENGTH} hours")

Python version: 3.14.4
Pandas version: 3.0.3
NumPy version: 2.5.1

Input file: ../Data/Interim/cleaned_air_quality.parquet
Output file: ../Data/Processed/engineered_air_quality.parquet

Encoder length: 168 hours
Prediction length: 15 hours


# 2. Load and Validate Cleaned Dataset

In [2]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {INPUT_FILE}\n"
        "Run 02_data_cleaning.ipynb first."
    )

df = pd.read_parquet(INPUT_FILE)

available_pollutants = [
    col
    for col in POLLUTANT_COLUMNS
    if col in df.columns
]

available_weather_columns = [
    col
    for col in WEATHER_COLUMNS
    if col in df.columns
]

missing_pollutants = [
    col
    for col in POLLUTANT_COLUMNS
    if col not in df.columns
]

required_columns = (
    ["timestamp"]
    + STATION_KEYS
    + POLLUTANT_COLUMNS
)

missing_required_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        "Required cleaned columns are missing: "
        f"{missing_required_columns}"
    )

if not pd.api.types.is_datetime64_any_dtype(
    df["timestamp"]
):
    raise TypeError(
        "timestamp must be datetime dtype."
    )

if df["timestamp"].isna().any():
    raise ValueError(
        "Invalid timestamps remain."
    )

duplicate_count = int(
    df.duplicated(
        subset=STATION_KEYS + ["timestamp"]
    ).sum()
)

if duplicate_count:
    raise ValueError(
        "Duplicate station-timestamp observations "
        f"remain: {duplicate_count:,}"
    )

print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {df.shape[1]:,}")

print(
    "Memory usage: "
    f"{df.memory_usage(deep=True).sum() / 1024**3:,.3f} GB"
)

print(
    f"Timestamp range: "
    f"{df['timestamp'].min()} "
    f"to {df['timestamp'].max()}"
)

print(
    f"Stations: "
    f"{df[STATION_KEYS].drop_duplicates().shape[0]:,}"
)

print(
    f"Available pollutants ({len(available_pollutants)}): "
    f"{available_pollutants}"
)

print(
    f"Missing pollutants ({len(missing_pollutants)}): "
    f"{missing_pollutants}"
)

print(
    f"Available weather columns: "
    f"{available_weather_columns}"
)

df.head()

Loaded rows: 3,429,120
Loaded columns: 18
Memory usage: 0.572 GB
Timestamp range: 2017-01-01 00:00:00 to 2026-06-30 23:00:00
Stations: 66
Available pollutants (9): ['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']
Missing pollutants (0): []
Available weather columns: ['ambient_temperature', 'relative_humidity', 'solar_radiation', 'rainfall']


,state,city,latitude,longitude,timestamp,pm2_5,pm10,no,no2,nox,nh3,so2,co,o3,ambient_temperature,relative_humidity,solar_radiation,rainfall
0,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,2017-09-05 11:00:00,25.0000,45.0000,1.8000,12.2000,7.9000,10.2000,5.6000,0.1000,79.5000,33.8000,69.0000,372.0000,0.0000
1,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,2017-09-05 12:00:00,25.0000,45.0000,1.8000,12.2000,7.9000,10.2000,5.6000,0.1000,79.5000,33.8000,69.0000,372.0000,0.0000
2,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,2017-09-05 13:00:00,25.0000,45.0000,1.8000,12.2000,7.9000,10.2000,5.6000,0.1000,79.5000,33.8000,69.0000,372.0000,0.0000
3,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,2017-09-05 14:00:00,25.0000,45.0000,1.8000,12.2000,7.9000,10.2000,5.6000,0.1000,79.5000,33.8000,69.0000,372.0000,0.0000
4,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,2017-09-05 15:00:00,23.0000,49.5000,0.6500,14.5500,8.2800,8.8500,4.5200,0.1500,62.5000,32.2200,70.5000,290.7500,0.0000


# 3. Sort time series and create stable station id

In [3]:
df = (
    df
    .sort_values(
        STATION_KEYS + ["timestamp"]
    )
    .reset_index(drop=True)
)

station_table = (
    df[STATION_KEYS]
    .drop_duplicates()
    .sort_values(STATION_KEYS)
    .reset_index(drop=True)
)

station_table["station_id"] = np.arange(
    len(station_table),
    dtype=np.int16,
)

df = df.merge(
    station_table,
    on=STATION_KEYS,
    how="left",
    validate="many_to_one",
)

df["station_id"] = df["station_id"].astype(
    np.int16
)

df = (
    df
    .sort_values(
        ["station_id", "timestamp"]
    )
    .reset_index(drop=True)
)

if df["station_id"].isna().any():
    raise ValueError(
        "Station ID assignment failed."
    )

if (
    df.groupby(
        "station_id",
        observed=True,
        sort=False,
    )["timestamp"]
    .apply(lambda series: series.is_monotonic_increasing)
    .eq(False)
    .any()
):
    raise ValueError(
        "Station-wise timestamps are not sorted."
    )

print(
    f"Stable station IDs created: "
    f"{df['station_id'].nunique():,}"
)

print("\nStation mapping sample:")

station_table.head(10)

Stable station IDs created: 66

Station mapping sample:


,state,city,latitude,longitude,station_id
0,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.9873,81.7363,0
1,Andhra Pradesh,"Gulzarpet, Anantapur (, )",14.6753,77.5970,1
2,Andhra Pradesh,"Gvm Corporation, Visakhapatnam (, )",17.7200,83.3000,2
3,Andhra Pradesh,"Secretariat, Amaravati (, )",16.5151,80.5182,3
4,Arunachal Pradesh,"Naharlagun, Naharlagun (, )",27.1030,93.7008,4
5,Assam,"Girls College, Sivasagar (, )",26.9892,94.6404,5
6,Assam,"Iitg, Guwahati (, )",26.1921,91.6951,6
7,Bihar,"Dm Office_Kasipur, Samastipur (, )",25.8598,85.7787,7
8,Bihar,"Gurdeo Nagar, Aurangabad (, )",24.7575,84.3662,8
9,Bihar,"Kamalnath Nagar, Bettiah (, )",26.8049,84.5057,9


# 4. Create global Hourly time index and gap features

In [4]:
GLOBAL_START_TIMESTAMP = df["timestamp"].min()

elapsed_hours = (
    (
        df["timestamp"]
        - GLOBAL_START_TIMESTAMP
    )
    .dt.total_seconds()
    .div(3600)
)

rounded_elapsed_hours = elapsed_hours.round()

non_hourly_timestamp_count = int(
    (
        ~np.isclose(
            elapsed_hours,
            rounded_elapsed_hours,
            rtol=0.0,
            atol=1e-9,
        )
    ).sum()
)

if non_hourly_timestamp_count:
    raise ValueError(
        "Non-hour-aligned timestamps detected: "
        f"{non_hourly_timestamp_count:,}"
    )

df["time_idx"] = (
    rounded_elapsed_hours
    .astype(np.int32)
)

station_groups = df.groupby(
    "station_id",
    observed=True,
    sort=False,
)

df["hours_since_previous"] = (
    station_groups["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(3600)
    .astype(np.float32)
)

df["has_time_gap"] = (
    df["hours_since_previous"]
    .gt(1.0)
    .fillna(False)
    .astype(np.int8)
)

negative_or_zero_gap_count = int(
    (
        df["hours_since_previous"]
        .dropna()
        <= 0
    ).sum()
)

if negative_or_zero_gap_count:
    raise ValueError(
        "Invalid non-positive station time gaps detected: "
        f"{negative_or_zero_gap_count:,}"
    )

print(
    f"Global start timestamp: "
    f"{GLOBAL_START_TIMESTAMP}"
)

print(
    f"Maximum time index: "
    f"{df['time_idx'].max():,}"
)

print(
    f"Rows following a time gap: "
    f"{df['has_time_gap'].sum():,}"
)

print("\nStation-wise timestamp gap summary:")

print(
    df["hours_since_previous"]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print("\nMost frequent time gaps:")

print(
    df["hours_since_previous"]
    .value_counts()
    .head(15)
)

Global start timestamp: 2017-01-01 00:00:00
Maximum time index: 83,231
Rows following a time gap: 8,843

Station-wise timestamp gap summary:
count   3,429,054.0000
mean            1.0868
std            11.1060
min             1.0000
50%             1.0000
90%             1.0000
95%             1.0000
99%             1.0000
max        11,156.0000
Name: hours_since_previous, dtype: float64

Most frequent time gaps:
hours_since_previous
1.0000     3420211
2.0000         878
5.0000         747
3.0000         554
4.0000         467
6.0000         363
8.0000         344
10.0000        329
11.0000        323
9.0000         320
7.0000         304
13.0000        299
12.0000        291
14.0000        278
15.0000        262
Name: count, dtype: int64


# 5. Create calender and cyclic time features

In [5]:
df["year"] = (
    df["timestamp"]
    .dt.year
    .astype(np.int16)
)

df["month"] = (
    df["timestamp"]
    .dt.month
    .astype(np.int8)
)

df["day"] = (
    df["timestamp"]
    .dt.day
    .astype(np.int8)
)

df["hour"] = (
    df["timestamp"]
    .dt.hour
    .astype(np.int8)
)

df["day_of_week"] = (
    df["timestamp"]
    .dt.dayofweek
    .astype(np.int8)
)

df["day_of_year"] = (
    df["timestamp"]
    .dt.dayofyear
    .astype(np.int16)
)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(np.int8)

TWO_PI = np.float32(2.0 * np.pi)

df["hour_sin"] = np.sin(
    TWO_PI * df["hour"] / 24.0
).astype(np.float32)

df["hour_cos"] = np.cos(
    TWO_PI * df["hour"] / 24.0
).astype(np.float32)

df["day_of_week_sin"] = np.sin(
    TWO_PI * df["day_of_week"] / 7.0
).astype(np.float32)

df["day_of_week_cos"] = np.cos(
    TWO_PI * df["day_of_week"] / 7.0
).astype(np.float32)

df["month_sin"] = np.sin(
    TWO_PI * (df["month"] - 1) / 12.0
).astype(np.float32)

df["month_cos"] = np.cos(
    TWO_PI * (df["month"] - 1) / 12.0
).astype(np.float32)

df["day_of_year_sin"] = np.sin(
    TWO_PI
    * (df["day_of_year"] - 1)
    / 365.25
).astype(np.float32)

df["day_of_year_cos"] = np.cos(
    TWO_PI
    * (df["day_of_year"] - 1)
    / 365.25
).astype(np.float32)

CALENDAR_COLUMNS = [
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "day_of_year",
    "is_weekend",
]

CYCLICAL_COLUMNS = [
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",
]

print("Calendar and cyclical features created.")

print(
    df[
        ["timestamp"]
        + CALENDAR_COLUMNS
        + CYCLICAL_COLUMNS
    ]
    .head()
    .to_string(index=False)
)

Calendar and cyclical features created.
          timestamp  year  month  day  hour  day_of_week  day_of_year  is_weekend  hour_sin  hour_cos  day_of_week_sin  day_of_week_cos  month_sin  month_cos  day_of_year_sin  day_of_year_cos
2017-09-05 11:00:00  2017      9    5    11            1          248           0    0.2588   -0.9659           0.7818           0.6235    -0.8660    -0.5000          -0.8945          -0.4470
2017-09-05 12:00:00  2017      9    5    12            1          248           0   -0.0000   -1.0000           0.7818           0.6235    -0.8660    -0.5000          -0.8945          -0.4470
2017-09-05 13:00:00  2017      9    5    13            1          248           0   -0.2588   -0.9659           0.7818           0.6235    -0.8660    -0.5000          -0.8945          -0.4470
2017-09-05 14:00:00  2017      9    5    14            1          248           0   -0.5000   -0.8660           0.7818           0.6235    -0.8660    -0.5000          -0.8945          -0.4470


# 6. Optimize Dtypes and define Model feature groups

In [6]:
memory_before = (
    df.memory_usage(deep=True).sum()
    / 1024**3
)

continuous_columns = (
    available_pollutants
    + available_weather_columns
    + [
        "latitude",
        "longitude",
        "hours_since_previous",
    ]
    + CYCLICAL_COLUMNS
)

for col in continuous_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        ).astype(np.float32)

df["station_id"] = df["station_id"].astype(
    np.int16
)

df["time_idx"] = df["time_idx"].astype(
    np.int32
)

df["state"] = df["state"].astype("category")
df["city"] = df["city"].astype("category")

TFT_STATIC_CATEGORICALS = [
    "station_id",
    "state",
]

TFT_STATIC_REALS = [
    "latitude",
    "longitude",
]

TFT_TIME_VARYING_KNOWN_REALS = [
    "time_idx",
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",
]

TFT_TIME_VARYING_OBSERVED_REALS = (
    available_pollutants
    + available_weather_columns
)

PATCHTST_INPUT_COLUMNS = (
    available_pollutants
    + available_weather_columns
    + CYCLICAL_COLUMNS
)

TARGET_COLUMNS = available_pollutants.copy()

memory_after = (
    df.memory_usage(deep=True).sum()
    / 1024**3
)

print(
    f"Memory before optimization: "
    f"{memory_before:,.3f} GB"
)

print(
    f"Memory after optimization: "
    f"{memory_after:,.3f} GB"
)

print(
    f"Memory reduction: "
    f"{memory_before - memory_after:,.3f} GB"
)

print("\nTFT static categoricals:")
print(TFT_STATIC_CATEGORICALS)

print("\nTFT static reals:")
print(TFT_STATIC_REALS)

print("\nTFT known future reals:")
print(TFT_TIME_VARYING_KNOWN_REALS)

print("\nTFT observed reals:")
print(TFT_TIME_VARYING_OBSERVED_REALS)

print("\nPatchTST input columns:")
print(PATCHTST_INPUT_COLUMNS)

print("\nForecast targets:")
print(TARGET_COLUMNS)

Memory before optimization: 0.738 GB
Memory after optimization: 0.390 GB
Memory reduction: 0.348 GB

TFT static categoricals:
['station_id', 'state']

TFT static reals:
['latitude', 'longitude']

TFT known future reals:
['time_idx', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']

TFT observed reals:
['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3', 'ambient_temperature', 'relative_humidity', 'solar_radiation', 'rainfall']

PatchTST input columns:
['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3', 'ambient_temperature', 'relative_humidity', 'solar_radiation', 'rainfall', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']

Forecast targets:
['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']


# 7. Analyses continous sequence availability

In [7]:
required_sequence_length = (
    ENCODER_LENGTH
    + PREDICTION_LENGTH
)

time_step_difference = (
    df.groupby(
        "station_id",
        observed=True,
        sort=False,
    )["time_idx"]
    .diff()
)

new_segment_mask = (
    time_step_difference.ne(1)
    | time_step_difference.isna()
)

df["sequence_segment_id"] = (
    new_segment_mask
    .groupby(df["station_id"])
    .cumsum()
    .astype(np.int32)
)

segment_lengths = (
    df.groupby(
        [
            "station_id",
            "sequence_segment_id",
        ],
        observed=True,
        sort=False,
    )
    .size()
    .rename("segment_length")
)

eligible_segments = (
    segment_lengths
    >= required_sequence_length
)

eligible_segment_count = int(
    eligible_segments.sum()
)

eligible_segment_rows = int(
    segment_lengths[
        eligible_segments
    ].sum()
)

estimated_valid_windows = int(
    (
        segment_lengths[
            eligible_segments
        ]
        - required_sequence_length
        + 1
    ).sum()
)

print(
    f"Required continuous sequence length: "
    f"{required_sequence_length} hours"
)

print(
    f"Total continuous segments: "
    f"{len(segment_lengths):,}"
)

print(
    f"Eligible continuous segments: "
    f"{eligible_segment_count:,}"
)

print(
    f"Rows inside eligible segments: "
    f"{eligible_segment_rows:,}"
)

print(
    f"Estimated valid 168→15 windows: "
    f"{estimated_valid_windows:,}"
)

print("\nContinuous segment length statistics:")

print(
    segment_lengths.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

if eligible_segment_count == 0:
    raise ValueError(
        "No continuous station sequence is long enough "
        f"for {ENCODER_LENGTH} encoder hours and "
        f"{PREDICTION_LENGTH} prediction hours."
    )

Required continuous sequence length: 183 hours
Total continuous segments: 8,909
Eligible continuous segments: 2,937
Rows inside eligible segments: 3,149,837
Estimated valid 168→15 windows: 2,615,303

Continuous segment length statistics:
count    8,909.0000
mean       384.9052
std      1,003.2347
min          1.0000
50%         69.0000
75%        308.0000
90%        938.2000
95%      1,726.2000
99%      4,998.6800
max     19,899.0000
Name: segment_length, dtype: float64


# 8. Final Validation

In [8]:
ENGINEERED_COLUMNS = (
    [
        "station_id",
        "time_idx",
        "hours_since_previous",
        "has_time_gap",
        "sequence_segment_id",
    ]
    + CALENDAR_COLUMNS
    + CYCLICAL_COLUMNS
)

assert not df.empty, (
    "Engineered dataset is empty."
)

assert df["timestamp"].notna().all(), (
    "Invalid timestamps remain."
)

assert df["station_id"].notna().all(), (
    "Missing station IDs remain."
)

assert df["time_idx"].notna().all(), (
    "Missing time indices remain."
)

assert df["sequence_segment_id"].notna().all(), (
    "Missing sequence segment IDs remain."
)

assert int(
    df.duplicated(
        subset=[
            "station_id",
            "timestamp",
        ]
    ).sum()
) == 0, (
    "Duplicate station-timestamp observations remain."
)

assert (
    df["time_idx"] >= 0
).all(), (
    "Negative time indices detected."
)

assert set(
    df["has_time_gap"]
    .dropna()
    .unique()
).issubset({0, 1}), (
    "Invalid has_time_gap values detected."
)

missing_engineered_columns = [
    col
    for col in ENGINEERED_COLUMNS
    if col not in df.columns
]

assert not missing_engineered_columns, (
    "Missing engineered columns: "
    f"{missing_engineered_columns}"
)

future_target_columns = [
    col
    for col in df.columns
    if (
        "_target_t+" in col
        or col.startswith("target_")
    )
]

assert not future_target_columns, (
    "Future target columns must not be "
    "created during feature engineering."
)

manual_lag_columns = [
    col
    for col in df.columns
    if "_lag_" in col
]

manual_rolling_columns = [
    col
    for col in df.columns
    if "_roll_" in col
]

assert not manual_lag_columns, (
    "Manual lag features detected."
)

assert not manual_rolling_columns, (
    "Manual rolling features detected."
)

station_monotonic = (
    df.groupby(
        "station_id",
        observed=True,
        sort=False,
    )["timestamp"]
    .apply(
        lambda series:
        series.is_monotonic_increasing
    )
)

assert bool(station_monotonic.all()), (
    "Station-wise timestamps are not "
    "monotonically increasing."
)

print("Final feature validation passed.")

print(f"\nFinal rows: {len(df):,}")
print(f"Final columns: {df.shape[1]:,}")

print(
    f"Stations: "
    f"{df['station_id'].nunique():,}"
)

print(
    f"Timestamp range: "
    f"{df['timestamp'].min()} "
    f"to {df['timestamp'].max()}"
)

print(
    f"Engineered columns: "
    f"{len(ENGINEERED_COLUMNS):,}"
)

print(
    f"Forecast targets: "
    f"{len(TARGET_COLUMNS)}"
)

Final feature validation passed.

Final rows: 3,429,120
Final columns: 38
Stations: 66
Timestamp range: 2017-01-01 00:00:00 to 2026-06-30 23:00:00
Engineered columns: 20
Forecast targets: 9


# 9. Inspect and save engineered dataset

In [9]:
inspection_columns = [
    "timestamp",
    "state",
    "city",
    "station_id",
    "time_idx",
    "sequence_segment_id",
    "hours_since_previous",
    "has_time_gap",
    "pm2_5",
    "pm10",
    "ambient_temperature",
    "hour",
    "day_of_week",
    "month",
    "hour_sin",
    "hour_cos",
]

existing_inspection_columns = [
    col
    for col in inspection_columns
    if col in df.columns
]

print("Engineered dataset sample:")

print(
    df[
        existing_inspection_columns
    ]
    .sample(
        10,
        random_state=42,
    )
    .sort_values(
        [
            "station_id",
            "timestamp",
        ]
    )
    .to_string(index=False)
)

try:
    import pyarrow as pa
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyArrow is required to save the "
        "engineered Parquet dataset. "
        "Install it with: pip install pyarrow"
    ) from exc

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

try:
    df.to_parquet(
        OUTPUT_FILE,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
except (
    pa.ArrowException,
    TypeError,
    ValueError,
) as parquet_error:
    print(
        "Parquet write failed. "
        "Environment diagnostics:"
    )

    print(
        f"Python version: "
        f"{sys.version}"
    )

    print(
        f"Pandas version: "
        f"{pd.__version__}"
    )

    print(
        f"PyArrow version: "
        f"{pa.__version__}"
    )

    raise parquet_error

if not OUTPUT_FILE.exists():
    raise FileNotFoundError(
        "Engineered feature dataset "
        "was not created."
    )

output_size_bytes = OUTPUT_FILE.stat().st_size
output_size_mb = output_size_bytes / 1024**2
output_size_gb = output_size_bytes / 1024**3

print(
    f"\nEngineered dataset saved to: "
    f"{OUTPUT_FILE}"
)

print(
    f"Parquet file size: "
    f"{output_size_mb:,.2f} MB"
)

print(
    f"Parquet file size: "
    f"{output_size_gb:,.4f} GB"
)

Engineered dataset sample:
          timestamp            state                               city  station_id  time_idx  sequence_segment_id  hours_since_previous  has_time_gap    pm2_5     pm10  ambient_temperature  hour  day_of_week  month  hour_sin  hour_cos
2023-08-16 16:00:00            Bihar Dm Office_Kasipur, Samastipur (, )           7     58048                   20                1.0000             0  30.7500  71.0000              32.8000    16            2      8   -0.8660   -0.5000
2022-03-13 07:00:00          Gujarat            Raikhad, Ahmedabad (, )          26     45535                   25                1.0000             0 129.5500 303.7300              24.4000     7            6      3    0.9659   -0.2588
2025-09-22 01:00:00 Himachal Pradesh Himuda Complex Phase-1, Baddi (, )          32     76465                   41                1.0000             0  60.7500 203.5000              27.5500     1            0      9    0.2588    0.9659
2021-10-06 07:00:00        Ka

# 10. Reload Verification and final report

In [10]:
saved_df = pd.read_parquet(
    OUTPUT_FILE,
)

assert len(saved_df) == len(df), (
    "Saved row count does not match "
    "the in-memory dataset."
)

assert list(saved_df.columns) == list(
    df.columns
), (
    "Saved column order does not match "
    "the in-memory dataset."
)

assert pd.api.types.is_datetime64_any_dtype(
    saved_df["timestamp"]
), (
    "Timestamp dtype changed after "
    "Parquet persistence."
)

assert saved_df["station_id"].dtype == df["station_id"].dtype, (
    "station_id dtype changed after persistence."
)

assert saved_df["time_idx"].dtype == df["time_idx"].dtype, (
    "time_idx dtype changed after persistence."
)

print("=" * 80)
print("FINAL FEATURE ENGINEERING REPORT")
print("=" * 80)

print(f"Rows: {len(saved_df):,}")
print(f"Columns: {saved_df.shape[1]:,}")

print(
    f"Stations: "
    f"{saved_df['station_id'].nunique():,}"
)

print(
    f"States: "
    f"{saved_df['state'].nunique():,}"
)

print(
    f"Timestamp range: "
    f"{saved_df['timestamp'].min()} "
    f"to {saved_df['timestamp'].max()}"
)

print(
    f"Encoder length: "
    f"{ENCODER_LENGTH} hours"
)

print(
    f"Prediction length: "
    f"{PREDICTION_LENGTH} hours"
)

print(
    f"Forecast targets ({len(TARGET_COLUMNS)}): "
    f"{TARGET_COLUMNS}"
)

print(
    f"Eligible continuous segments: "
    f"{eligible_segment_count:,}"
)

print(
    f"Estimated valid sequence windows: "
    f"{estimated_valid_windows:,}"
)

print(
    f"Output file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Output size: "
    f"{output_size_mb:,.2f} MB "
    f"({output_size_gb:,.4f} GB)"
)

print("\nFeature engineering decisions:")

print(
    "- No manual lag features"
)

print(
    "- No manual rolling features"
)

print(
    "- No future target columns"
)

print(
    "- No scaling before chronological split"
)

print(
    "- No interpolation or backward filling"
)

print(
    "- Continuous hourly segments identified"
)

print(
    "- Dataset prepared for TFT and PatchTST"
)

print("=" * 80)

del saved_df
gc.collect()

FINAL FEATURE ENGINEERING REPORT
Rows: 3,429,120
Columns: 38
Stations: 66
States: 19
Timestamp range: 2017-01-01 00:00:00 to 2026-06-30 23:00:00
Encoder length: 168 hours
Prediction length: 15 hours
Forecast targets (9): ['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']
Eligible continuous segments: 2,937
Estimated valid sequence windows: 2,615,303
Output file: ../Data/Processed/engineered_air_quality.parquet
Output size: 97.32 MB (0.0950 GB)

Feature engineering decisions:
- No manual lag features
- No manual rolling features
- No future target columns
- No scaling before chronological split
- No interpolation or backward filling
- Continuous hourly segments identified
- Dataset prepared for TFT and PatchTST


0